# 검색 최적화를 위한 Indexing

Indexing(인덱싱)은 외부 문서를 검색할 수 있는 형태로 변환해 저장하는 준비 과정이다. 이 실습에서는 공개된 `documents.csv`를 직접 내려받고, 각 본문을 OpenAI 임베딩 벡터로 변환해 Pinecone에 저장한다.

이름이 비슷한 세 용어의 역할은 다음과 같다.

- **Indexing**: 문서 준비부터 임베딩과 저장까지 수행하는 과정이다.
- **Pinecone index**: 벡터, 문서 식별자와 메타데이터가 저장되는 원격 검색 공간이다.
- **PineconeVectorStore**: LangChain 코드에서 Pinecone index의 저장과 검색 기능을 호출하는 연결 객체이다.

전체 데이터 흐름은 `CSV 행 → Document → embedding vector → Pinecone index → 유사 문서와 점수`이다. 

같은 `doc_id`를 Pinecone 레코드 ID로 사용하면 노트북을 다시 실행해도 같은 문서를 갱신할 수 있다.

## Indexing 실행 패키지 준비



In [ ]:
# %pip install -U pandas langchain langchain-openai langchain-pinecone pinecone python-dotenv gdown

## API key 환경 변수 준비

`load_dotenv()`로 `.env`의 OpenAI·Pinecone API key를 환경 변수에 등록한다. 키 값은 출력하지 않으며 뒤의 `Pinecone()`과 `OpenAIEmbeddings`가 필요한 인증값을 직접 읽는다.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=False)

True

## Pinecone index와 임베딩 설정

`PINECONE_NAMESPACE`는 하나의 index 안에서 레코드 묶음을 논리적으로 분리하는 이름이다. 값이 없으면 Pinecone의 기본 namespace인 빈 문자열을 사용한다. 이후 BM25·Dense 비교와 RRF가 같은 벡터를 검색하려면 세 노트북이 같은 index 이름과 namespace를 사용해야 한다.

기본 설정은 `text-embedding-3-small`의 1,536차원 벡터를 사용하고, cosine similarity score를 해석하기위해 Pinecone metric은 `cosine`만 허용한다.



In [3]:
PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME', 'adv-rag')
PINECONE_NAMESPACE = os.getenv('PINECONE_NAMESPACE', '').strip()
PINECONE_INDEX_CLOUD = os.getenv('PINECONE_INDEX_CLOUD', 'aws')
PINECONE_INDEX_REGION = os.getenv('PINECONE_INDEX_REGION', 'us-east-1')
PINECONE_INDEX_METRIC = os.getenv(
    'PINECONE_INDEX_METRIC', 'cosine'
).strip().lower()
PINECONE_INDEX_DIMENSION = int(os.getenv('PINECONE_INDEX_DIMENSION', '1536'))
OPENAI_EMBEDDING_MODEL = os.getenv(
    'OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'
).strip() or 'text-embedding-3-small'

## Pinecone index 생성 또는 재사용

`Pinecone()`은 `.env`의 `PINECONE_API_KEY`를 이용해 Pinecone 관리 API에 연결한다. 

`create_index()`의 주요 인자는 다음 역할을 가진다.

- `name`: 저장과 검색에 사용할 index 식별자이다.
- `dimension`: 한 벡터의 숫자 개수이며 임베딩 출력 차원과 같아야 한다.
- `metric`: 벡터 사이의 가까움을 계산하는 방식이다.
- `ServerlessSpec`: index를 배치할 cloud와 region을 지정한다.

같은 이름이 없을 때만 index를 생성하고, 이미 있으면 재사용한다. 

생성 요청은 index 준비보다 먼저 끝날 수 있으므로 다음 셀에서 `ready` 상태를 확인한다. 

[Pinecone index 관리 문서](https://docs.pinecone.io/guides/manage-data/manage-indexes)


In [4]:
from grpc import Server
from pinecone import Pinecone, ServerlessSpec

# 1. Pinecone 연결 객체 생성
# api_key 생략 시 환경 변수에서 PINECONE_API_KEY를 자동으로 찾아 등록
pc = Pinecone()

# 해당 사용자가 보유하고 있는 index 목록 반환
existing_indexes = pc.list_indexes().names()
print(existing_indexes)

# 2. Pinecone의 ServerLess index 생성 요청 보내기
# 단, 같은 이름이 index가 없을 경우에만 가능하다
# if PINECONE_INDEX_NAME not in existing_indexes:
if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        # 이름 : adv-rag
        name=PINECONE_INDEX_NAME,
        # 벡터타입
        metric=PINECONE_INDEX_METRIC,
        # 디멘션 개수 : 1536
        dimension=PINECONE_INDEX_DIMENSION,
        # 스팩
        spec=ServerlessSpec(
            # 클라우드
            cloud=PINECONE_INDEX_CLOUD,
            # 지역
            region=PINECONE_INDEX_REGION,
        ),
    )
    print(f"Index {PINECONE_INDEX_NAME} created successfully.")
    
    
else :
  print(f"Index {PINECONE_INDEX_NAME} already exists.")

['adv-rag', 'winemag-review-data', 'pinecone-first']
Index adv-rag already exists.


## index가 적재 가능한 상태인지 확인

`create_index()`는 생성 요청을 보낸 뒤 실제 준비가 끝나기 전에 반환될 수 있다. 다음 코드는 `ready=True`가 될 때까지 기다린 뒤 바로 Vector Store 연결 단계로 넘어간다.


In [5]:
from time import sleep

while not pc.describe_index(name=PINECONE_INDEX_NAME).status['ready']:
    sleep(1)
    
print("index ready : ", PINECONE_INDEX_NAME)

index ready :  adv-rag


## 임베딩 모델과 PineconeVectorStore 연결

`OpenAIEmbeddings`는 텍스트를 숫자 벡터로 변환하는 객체이다. `PineconeVectorStore`는 Pinecone 자체가 아니라, LangChain에서 Pinecone index에 문서를 저장하고 검색하기 위한 연결 객체이다.

- `model`: 텍스트를 어떤 방식과 가중치로 벡터화할지 정하는 임베딩 모델이다.
- `dimensions`: 생성할 문서·query 벡터의  숫자 개수이다. Pinecone index의 `dimension`과 같아야 한다.
- `index_name`: 앞에서 준비한 Pinecone index를 선택한다.
- `embedding`: 저장할 본문과 검색 질문을 같은 벡터 공간으로 변환한다.
- `namespace`: 해당 index 안에서 저장과 검색이 사용할 동일한 레코드 영역을 선택한다.

문서와 질문에는 같은 `model`과 `dimensions`를 사용한다. 생성된 벡터 길이가 Pinecone index dimension과 다르거나 저장과 검색의 namespace가 다르면 정상적으로 저장·검색할 수 없다. [LangChain Pinecone 연동 문서](https://docs.langchain.com/oss/python/integrations/vectorstores/pinecone)


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    dimensions=PINECONE_INDEX_DIMENSION,
)

vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,
)

## documents.csv 다운로드

Google Drive에 공개된 수업용 `documents.csv`를 내려받는다. `gdown`의 파일 ID는 실제 문서 데이터 위치를 가리키고, `-O documents.csv`는 다음 `pd.read_csv()` 셀이 사용할 로컬 파일명을 정한다. 이 셀은 네트워크 다운로드를 수행하므로 수업 환경에서 인터넷 연결을 확인한 뒤 실행한다.


In [ ]:
# !gdown 1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k -O documents.csv

Downloading...
From: https://drive.google.com/uc?id=1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k
To: /Users/choejiheum/SK_AI/09_llm/07_advenced_rag/01_retrieval_optimization/documents.csv
100%|█████████████████████████████████████| 14.9k/14.9k [00:00<00:00, 18.6MB/s]


## CSV 읽기

Pandas의 `pd.read_csv()`는 내려받은 `documents.csv`를 행과 열로 구성된 `DataFrame`으로 읽는다. 한 행이 뒤에서 하나의 `Document`가 된다.

먼저 전체 shape와 열 이름을 출력하고  문서 세 건을 확인한다. 이어지는 셀에서 이 표가 인덱싱에 필요한 schema를 만족하는지 검사한다.


In [9]:
import pandas as pd

document_df = pd.read_csv("documents.csv")
print("document_df : ", document_df.shape)
print("Columns : ", document_df.columns.tolist())

display(document_df.head())

document_df :  (30, 3)
Columns :  ['doc_id', 'title', 'content']


,doc_id,title,content
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(..."
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기..."
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet..."
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...


## DataFrame 행을 인덱싱 입력으로 변환

`iterrows()`는 DataFrame을 한 행씩 순회하며 `(행 index, Series)`를 반환한다. 여기서는 행 번호가 필요하지 않으므로 `_`로 받고, 각 `Series`에서 `doc_id`와 `content`를 꺼낸다.

변환 결과인 `(문서 ID, 본문)` 튜플 목록은 다음 셀에서 `Document` 목록과 Pinecone 레코드 ID 목록으로 분리된다.


In [12]:
docs_to_index = []

# iterrows(): Dataframe의 한 줄을 (행 번호, 행) 형태로 반환
for _, row in document_df.iterrows():
    # print("_ : ", _, "row : ", row)
    # doc_to_index.append(row)
    doc_id = str(row["doc_id"])
    content = str(row["content"])
    docs_to_index.append((doc_id, content))
    
print("docs_to_index : ", len(docs_to_index))
print("first document id : ", docs_to_index[0][0])
print("first document content : ", docs_to_index[0][1])

docs_to_index :  30
first document id :  D1
first document content :  제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다.


## Document 생성과 고정 ID upsert

`Document`는 LangChain이 문서 본문과 부가 정보를 함께 다루는 표준 객체이다.

- `page_content`: OpenAI 임베딩 모델에 전달되는  본문이다.
- `metadata`: 벡터와 함께 저장되며 검색 뒤 문서 ID를 복원한다.
- `ids`: Pinecone에서 각 레코드를 구분하는 고정 ID 목록이다.

`add_documents()`를 호출하면 `page_content → OpenAI embedding → Pinecone vector upsert`가 실행된다. Upsert는 ID가 없으면 새 레코드를 만들고, 같은 ID가 있으면 그 레코드를 갱신하는 저장 방식이다. 문서 목록과 ID 목록을 한 번에 전달해 CSV의 문서 30건을 `PINECONE_NAMESPACE` 영역에 batch로 처리한다.


In [ ]:
from langchain_core.documents import Document

# 1. documents와 document_ids list 생성
documents = []
documents_ids = []

# 2. docs_to_index의 요소를 얻어와 LangChain Document 변환
# + 1번에서 생성한 list의 누적
for doc_id, content in docs_to_index:
    document = Document(
        # 내용
        page_content=content,
        # 메타데이터 입력
        metadata={"doc_id": doc_id},
    )

    documents.append(document)
    documents_ids.append(doc_id)
    # 각 list의 같은 인덱스 요소가 한 세트를 이루게 만듦.

# 3. Vector store에 Documents추가
# - 단순 insert를 실항하는게 아닌
# -> 고정 id를 제공해서 Vector Store에 같은 id가 없다면 insert
#    고정되는 id가 있드면 update를 하는 upsert진행
upsert_ids = vector_store.add_documents(
    documents=documents,
    ids=documents_ids,
)

print("입력된 Document 수 : ", len(documents))
print("upsert 결과 로 반환된 id 수 : ", len(upsert_ids))

입력된 Document 수 :  30
upsert 결과 로 반환된 id 수 :  30


## 질문 임베딩과 상위 5개 검색

`similarity_search_with_score()`는 질문을 문서와 같은 OpenAI 모델로 임베딩한 뒤 `vector_store`에 설정된 동일한 namespace에서 가까운 벡터를 찾는다.

- `query`: 검색할 자연어 질문이다.
- `k=5`: 최대 다섯 건을 반환한다.
- 반환값: 유사한 순서의 `list[tuple[Document, float]]`이다. 각 튜플의 첫 값은 원문과 메타데이터를 가진 `Document`, 둘째 값은 cosine similarity score이다.

현재 `cosine` 구성에서는 score가 큰 문서부터 반환되며, 높은 값일수록 질문과 더 유사하다. 이 score는 모델이 정답일 확률이 아니라 두 embedding vector가 얼마나 같은 방향을 가리키는지 나타내는 값이다.


In [ ]:
results: list[tuple[Document, float]] = vector_store.similarity_search_with_score(
    # 검색어
    # 코사인 유사도 방식은
    # OpenAI Embedding Vector의 1534개의 벡터로 변환후 코사인 유사도 비슷한것을 찾아 결과를 나타낸다
    query="제주도 관광지",
    # 상위 5개
    k=5,
)

for rank, (doc, score) in enumerate(results):
    print(f"{rank} : {doc.metadata['doc_id']}")
    print(f"cosine similarity : {score:.4f}")

    print(doc.page_content)
    print("")

0 : D1
cosine similarity : 0.5898
제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다.

1 : D12
cosine similarity : 0.3095
서울 근교에서 당일치기로 다녀올 만한 여행지로는 가평 쁘띠프랑스, 남양주 수종사, 양평 두물머리, 용인 에버랜드 등이 있습니다. 기차·버스 노선이 잘 발달되어 있어 대중교통으로 이동이 편리하며, 차가 있다면 경춘고속도로를 이용해 접근성이 좋습니다. 사전 관광 예약 앱(예: 야놀자, 쿠팡트래블)에서도 할인 혜택을 확인할 수 있습니다.

2 : D13
cosine similarity : 0.2315
비빔밥은 지역별로 칼로리, 탄수화물, 단백질, 지방 함량이 차이를 보입니다. 전주 비빔밥(약 650kcal)은 채소·고기·계란 비율이 고르지만, 진주 비빔밥(약 700kcal)은 해산물과 육류가 섞여 열량이 다소 높습니다. 안동 비빔밥은 재료가 비교적 간단해 600kcal 내외이며, 지역별 나물 종류와 기름 사용량이 칼로리 차이에 영향을 미칩니다.

3 : D2
cosine similarity : 0.2280
비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기·계란 등)을 올리고 고추장이나 간장을 섞어 먹습니다. 전주 비빔밥은 고명 종류가 다양하고 전주식 고추장을 쓰며, 잔치용으로도 유명합니다. 진주 비빔밥은 고기·회·나물 등을 섞어 더욱 풍부한 식감을 제공합니다. 두 지역 모두 역사적 배경과 재료 구성이 달라 맛과 풍미가 다릅니다.

4 : D8
cosine similarity : 0.1972
서울 지하철은 1호선부터 9호선까지 운행되며, 주요 환승역으로는 서울역·강남역·종로